## Here is a short script to generate text lists to control the scan parameters

In [2]:
import os
import sys

if '../' not in sys.path:
    sys.path.append('../')

In [3]:
import numpy as np

In [4]:
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

In [5]:
masses = np.geomspace(100, 1e6)

## define the scan range for m_de

In [6]:
mass_ratios = np.array([1.0, 1000.0]) 

num_mdp_per_mde = len(mass_ratios)

m_de_range = masses
m_dp_ranges = np.zeros((len(m_de_range), num_mdp_per_mde))


In [7]:
for (i, m_de) in enumerate(m_de_range):
    for j in range(num_mdp_per_mde):
        m_dp_ranges[i, j] = m_de*mass_ratios[j]

rate_masses = m_dp_ranges.flatten()

In [8]:
coulomb_mass_list_name = 'coulomb_mass_list_highmass.txt'
cml_path = os.path.join('../input/', coulomb_mass_list_name)

with open(cml_path, 'w') as out_file:
    for m in rate_masses:
        print(f'{m:.3e}', file=out_file)

In [9]:
ann_mass_list_name = 'annihilation_masses_highmass.csv'
aml_path = os.path.join('../input/', ann_mass_list_name)

with open(aml_path, 'w') as out_file:
    for m in rate_masses:
        print(f'{m:.3e}, {1e-2*m:.3e}, {1e5*m:.3e}', file=out_file)

In [10]:
rate_masses_temp = np.zeros(len(rate_masses))
for (i, rm) in enumerate(rate_masses):
    rate_masses_temp[i] = float(f'{rm:.3e}')

In [11]:
adm_scan_file = 'adm_Neff_scan_highmass.txt'
adms_path = os.path.join('../input/', adm_scan_file)

eps_min = 1e-10
eps_max = 1e-4

num_eps = 50


# heres how to make all the parameter points as a flattened array
N_param_points = len(m_de_range)*num_mdp_per_mde*num_eps

print(N_param_points)

flattened_scan = np.zeros((N_param_points, 3))

index = 0

for (i, m_de) in enumerate(m_de_range):
    for j in range(num_mdp_per_mde):
        m_dp = m_dp_ranges[i,j]
        for (k, eps) in enumerate(np.geomspace(eps_min, eps_max, num_eps)):
            flattened_scan[index, 0] = m_de
            flattened_scan[index, 1] = m_dp
            flattened_scan[index, 2] = eps
            index += 1


with open(adms_path, 'w') as out_file:
    np.savetxt(out_file, flattened_scan, fmt='%.3e', delimiter=',', header='m_de [MeV], m_dp [MeV], epsilon')


5000


In [12]:
flattened_scan[0, 0]

100.0

In [13]:
m_de_range[0]

100.0

## Check for corrupted npz files

In [17]:
check_dir = '../output/rates/coulomb/cluster/scan/'

In [18]:
bad_files = []
for f in os.listdir(check_dir):
    if not f.endswith('.npz'):
        continue
    test = np.load(os.path.join(check_dir, f))
    key_0 = list(test.keys())[0]
    
    try:
        foo = test[key_0]

    except:
        bad_files.append(f)

In [19]:
bad_files

[]

In [177]:
for bad_file in bad_files:
    os.remove(os.path.join(check_dir, bad_file))